# PDF Page/Paragraph Chunking Example (pypdf + LLM)

PDF를 페이지 단위로 읽고, 각 페이지를 LLM에 보내 문단 단위로 분할하는 예제입니다.


In [1]:
from pathlib import Path
import json
import re

from pypdf import PdfReader
from langchain_openai import ChatOpenAI
from tqdm import tqdm

In [2]:
# 순차적으로 읽을 PDF 목록
PDF_DIR = Path("Papers")
pdf_paths = sorted(PDF_DIR.glob("*.pdf"))

print(f"Found {len(pdf_paths)} PDF files")
for p in pdf_paths[:10]:
    print("-", p)


Found 3 PDF files
- Papers/2408.06361v2.pdf
- Papers/2504.10789v1.pdf
- Papers/3582560.pdf


In [3]:
def split_paragraphs_rule_based(page_text: str) -> list[str]:
    """LLM 실패 시 사용하는 규칙 기반 문단 분리기."""
    if not page_text:
        return []

    text = page_text.replace('\r\n', '\n').replace('\r', '\n')
    blocks = re.split(r'\n\s*\n+', text)

    paragraphs = []
    for block in blocks:
        cleaned = re.sub(r'[ \t]+', ' ', block).strip()
        if cleaned:
            paragraphs.append(cleaned)
    return paragraphs


def _extract_json_array(text: str) -> list[str]:
    """LLM 응답에서 JSON 배열을 찾아 파싱한다."""
    candidate = text.strip()

    # 코드펜스 제거
    candidate = re.sub(r'^```(?:json)?\s*', '', candidate)
    candidate = re.sub(r'\s*```$', '', candidate)

    # 전체가 JSON이 아니면 첫 [ ... ] 블록 탐색
    if not (candidate.startswith('[') and candidate.endswith(']')):
        m = re.search(r'\[[\s\S]*\]', candidate)
        if not m:
            raise ValueError('JSON array not found in LLM response')
        candidate = m.group(0)

    data = json.loads(candidate)
    if not isinstance(data, list):
        raise ValueError('Parsed JSON is not a list')

    out = []
    for item in data:
        if isinstance(item, str):
            t = item.strip()
            if t:
                out.append(t)
    return out


def split_paragraphs_with_llm(
    page_text: str,
    llm,
    max_chars: int = 12000,
    max_paragraphs: int = 10,
) -> list[str]:
    if not page_text:
        return []

    text = page_text[:max_chars]

    prompt = f"""
You split PDF text into paragraphs.
Return ONLY a JSON array of strings.
Rules:
- Keep original language and wording.
- Do not summarize or rewrite.
- Merge wrapped lines into natural paragraphs.
- Keep headings as separate paragraph items when appropriate.
- Do not include any keys, markdown, or explanation.

TEXT:
{text}
"""

    try:
        response = llm.invoke(prompt)
        content = response.content if hasattr(response, 'content') else str(response)

        paragraphs = _extract_json_array(content)
        if paragraphs:
            return paragraphs[:max_paragraphs]
    except Exception:
        # API 실패, 응답 파싱 실패 등은 규칙 기반 방식으로 fallback
        pass

    return split_paragraphs_rule_based(page_text)[:max_paragraphs]


In [4]:
# 1) 실행 파라미터 설정
MODEL = "google/gemma-3-4b"
API_BASE = "http://host.docker.internal:1234/v1"
API_KEY = "lm-studio"
TEMPERATURE = 0.2
MAX_TOKENS = 800
MAX_PARAGRAPHS = 10
MAX_PAGE_CHARS_FOR_LLM = 12000

# 2) LLM 클라이언트 생성
llm = ChatOpenAI(
    model=MODEL,
    base_url=API_BASE,
    api_key=API_KEY,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

# 3) PDF 페이지 -> 문단(LLM)
# 모든 PDF가 아니라, 목록에서 파일 1개를 임의 선택해 처리
import random

page_chunks = []
paragraph_chunks = []

if not pdf_paths:
    raise ValueError('No PDF files found')

selected_pdf = random.choice(pdf_paths)
print(f'Selected PDF: {selected_pdf}')

reader = PdfReader(str(selected_pdf))

for page_num, page in tqdm(enumerate(reader.pages, start=1), total=len(reader.pages)):
    page_text = (page.extract_text() or '').strip()
    if not page_text:
        continue

    page_chunks.append({
        'file': str(selected_pdf),
        'page': page_num,
        'text': page_text,
    })

    paragraphs = split_paragraphs_with_llm(
        page_text=page_text,
        llm=llm,
        max_chars=MAX_PAGE_CHARS_FOR_LLM,
        max_paragraphs=MAX_PARAGRAPHS,
    )

    for para_idx, para_text in enumerate(paragraphs, start=1):
        paragraph_chunks.append({
            'file': str(selected_pdf),
            'page': page_num,
            'paragraph': para_idx,
            'text': para_text,
        })

print(f'Total page chunks: {len(page_chunks)}')
print(f'Total paragraph chunks: {len(paragraph_chunks)}')


Selected PDF: Papers/2408.06361v2.pdf


100%|██████████| 8/8 [03:28<00:00, 26.12s/it]

Total page chunks: 8
Total paragraph chunks: 15


In [7]:
print(paragraph_chunks[10]['text'])

Xue Zhang, Hauke Fuehres, and Peter A. Gloor. 2011. Predicting Stock Market Indicators Through Twitter “I hope it is not as bad as I fear”.Procedia - Social and Behavioral Sciences26 (2011), 55–62. https://doi.org/10.1016/j.sbspro.2011.10.562


In [5]:
# 샘플 출력
print("[Page chunk sample]")
if page_chunks:
    sample_page = page_chunks[0]
    print(sample_page["file"], "p.", sample_page["page"])
    print(sample_page["text"][:500])
else:
    print("No page chunks")

print("\n[Paragraph chunk sample]")
if paragraph_chunks:
    sample_para = paragraph_chunks[0]
    print(sample_para["file"], "p.", sample_para["page"], "para", sample_para["paragraph"])
    print(sample_para["text"][:500])
else:
    print("No paragraph chunks")


[Page chunk sample]
Papers/2408.06361v2.pdf p. 1
Large Language Model Agent in Financial Trading: A Survey
Han Ding∗
hd2412@columbia.edu
Columbia University
New York, NY, USA
Yinheng Li∗
yl4039@columbia.edu
Columbia University
New York, NY, USA
Junhao Wang∗
jw3668@columbia.edu
Columbia University
New York, NY, USA
Hang Chen∗
hc2798@nyu.edu
New York University
New York, NY, USA
Doudou Guo∗
dg3195@columbia.edu
Columbia University
New York, NY, USA
Yunbai Zhang∗
yz3386@columbia.edu
Columbia University
New York, NY, USA
ABSTRACT
Trading is a highl

[Paragraph chunk sample]
Papers/2408.06361v2.pdf p. 1 para 1
Large Language Model Agent in Financial Trading: A Survey
Han Ding∗
hd2412@columbia.edu
Columbia University
New York, NY, USA
Yinheng Li∗
yl4039@columbia.edu
Columbia University
New York, NY, USA
Junhao Wang∗
jw3668@columbia.edu
Columbia University
New York, NY, USA
Hang Chen∗
hc2798@nyu.edu
New York University
New York, NY, USA
Doudou Guo∗
dg3195@columbia.edu
Columbia University
New Y